### 문제 1-1 : 기본 Chain 만들기 - AI 요리사

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template(
    "다음 재료들로 만들 수 있는 요리를 추천주고, 레시피를 알려주세요: {ingredients}"
)

model = ChatOpenAI(temperature=0.7, seed=42)

parser = StrOutputParser()

chain = prompt | model | parser

result = chain.invoke({"ingredients": "계란, 밥, 김치"})

print(result)

### 문제 1-2 : 2단계 Multi Chain 만들기 - 여행지 정보 시스템

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(temperature=0.7)
parser = StrOutputParser()

# TODO 1: 체인1용 프롬프트 — 여행지를 입력받아 대표 명소 1곳만 추천
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "여행 전문가"),
    ("user", "{destination} 여행지에 대해 대표 명소 1곳만 추천해 주세요.")
])

# TODO 2: 체인1 연결 (LCEL)
chain1 = prompt1 | model | parser

# TODO 3: 체인1 실행하고 결과를 변수에 저장 + 출력
step1_result = chain1.invoke({"destination": "로마"})
print("1단계 결과:", step1_result)

# TODO 4: 체인2용 프롬프트 — 명소 이름을 받아 상세 정보(역사, 특징, 방문 팁) 제공
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "여행 전문가"),
    ("user", "다음 명소에 대해 자세한 정보를 제공해 주세요: {attraction}")
])

# TODO 5: 체인2 연결 (LCEL)
chain2 = prompt2 | model | parser

# TODO 6: 체인2 실행 — 이때 넘기는 값이 바로 step1_result 여야 합니다
step2_result = chain2.invoke({"attraction": step1_result})
print("2단계 결과:", step2_result)

### 문제 1-3 : FewShotPromptTemplate과 시스템 메시지 활용 

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    FewShotChatMessagePromptTemplate,
    ChatPromptTemplate
)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}")
])

# TODO 1: 예시를 최소 3개로 만드세요 (지금은 2개뿐이에요)
examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다...",
        "keywords": "삼성전자, 인공지능, 엔비디아"
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다...",
        "keywords": "세계보건기구, 건강위기, 국제협력"
    },
    {
        "news": "대한민국이 아시안게임에서 금메달을 획득하며 뛰어난 성과를 보였다.",  # TODO: 다른 분야 뉴스 본문을 직접 써보세요
        "keywords": "아시안게임, 금메달, 대한민국"  # TODO: 그 뉴스의 핵심 키워드 3개
    }
]

# 2. Few-Shot 프롬프트
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# 3. 최종 프롬프트
# TODO 2: 마지막 ("human", ...) 자리에 실제 테스트 뉴스가 들어갈 변수명을 정하세요
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "뉴스 키워드 추출 전문가입니다. 핵심 키워드 3개를 추출하세요."),
    few_shot_prompt,
    ("human", "{input_news}")  # TODO: 테스트 뉴스가 들어갈 변수명
])

# TODO 3: 모델과 파서를 만드세요 (1-1, 1-2에서 했던 것과 같아요)
model = ChatOpenAI(temperature=0.7, seed=42)
parser = StrOutputParser()

# TODO 4: LCEL로 체인을 연결하세요
chain = final_prompt | model | parser

# TODO 5: 테스트 뉴스로 체인을 실행하세요
test_news = "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다..."
result = chain.invoke({"input_news": test_news})
print(result)